# Model Prediction

## Import necessary libraries

In [ ]:
%pip install -qq -r ../requirements.txt

In [ ]:
# Add current directory to Python path for imports
import os
import sys

# Add the parent directory (project root) to Python path so we can import from src
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
# Utility Functions
from src.utils import create_spark_session

# Create Spark session
spark, sedona = create_spark_session(app_name="ModelPredictionSpark")

## Loading Datasets

In [ ]:
from src.utils import read_config_path

# Load data using configuration file
filepath = read_config_path(key="raw_data_path")

df = spark.read.csv(
    filepath,
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"',
    quote='"',
)

df.show(10)

## Sample data

In [ ]:
from pyspark.sql import functions as F

sample_df = df.limit(100)
sample_df.show(10)

---

## Applying Cleansing Pipeline

In [ ]:
from src.pipelines_spark import CleansingPipelineSpark

cleansing_pipeline = CleansingPipelineSpark(spark, sedona)
df_cleansed = cleansing_pipeline.transform(sample_df)

df_cleansed.show(10)

In [ ]:
df_cleansed.printSchema()

---

## Applying Model Preparation Pipeline

In [ ]:
from src.pipelines_spark import ModelPrepPipelineSpark

preparing_pipeline = ModelPrepPipelineSpark()
df_prepared = preparing_pipeline.transform(df_cleansed)

df_prepared.show(10)

In [ ]:
df_prepared.printSchema()

---

## Predict prepared data

In [ ]:
from src.utils import predict_with_model

model_path = "path/to/model"

pred_df = predict_with_model(
    spark=spark,
    model_path=model_path,
    input_df=df_prepared,
)

pred_df.show(10)

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol="resolution_time", predictionCol="prediction", metricName="rmse"
)

rmse = evaluator.evaluate(pred_df)
print("RMSE:", rmse)

In [ ]:
spark.stop()

---